<a href="https://colab.research.google.com/github/shoxruxmamitov/machine-learning1/blob/main/machine_learning_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import sklearn

url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(url)

In [2]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

x_train = train_set.drop("median_house_value", axis=1)
y = train_set['median_house_value'].copy()

x_num = x_train.drop("ocean_proximity", axis=1)

pepeline qurish

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room = True):
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
            return self
    def transform(self, x):
        rooms_per_household = x[:, rooms_ix] / x[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = x[:, bedrooms_ix] / x[:, rooms_ix]
            return np.c_[x, rooms_per_household, bedrooms_per_room]
        else:
             return np.c_[x, rooms_per_household]


sonli ustunlar uchun

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('attribs_adder', CombinedAttributesAdder(add_bedrooms_per_room = True)),
    ('std_scaler', StandardScaler())
])

matnli ustunlar uchun

In [6]:
from sklearn.compose import ColumnTransformer

num_attribs = list(x_num)
cat_attribs = ['ocean_proximity']

full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_attribs),
    ('cat', OneHotEncoder(), cat_attribs)
])

In [7]:
X_prepared = full_pipeline.fit_transform(x_train)

In [8]:
X_prepared

array([[ 1.27258656, -1.3728112 ,  0.34849025, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.70916212, -0.87669601,  1.61811813, ...,  0.        ,
         0.        ,  1.        ],
       [-0.44760309, -0.46014647, -1.95271028, ...,  0.        ,
         0.        ,  1.        ],
       ...,
       [ 0.59946887, -0.75500738,  0.58654547, ...,  0.        ,
         0.        ,  0.        ],
       [-1.18553953,  0.90651045, -1.07984112, ...,  0.        ,
         0.        ,  0.        ],
       [-1.41489815,  0.99543676,  1.85617335, ...,  0.        ,
         1.        ,  0.        ]])

linear regression

In [9]:
from sklearn.linear_model import LinearRegression

LR_model = LinearRegression()
LR_model.fit(X_prepared, y)

LinearRegression()

In [10]:
test_data = x_train.sample(10)
test_data

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
14983,-116.99,32.72,13.0,1330.0,216.0,719.0,215.0,3.8295,<1H OCEAN
8289,-118.14,33.76,37.0,3242.0,698.0,1080.0,629.0,3.9010,NEAR OCEAN
10235,-117.92,33.86,26.0,745.0,161.0,247.0,151.0,3.6375,<1H OCEAN
15301,-117.36,33.18,39.0,1546.0,291.0,833.0,308.0,2.8893,NEAR OCEAN
9880,-121.79,36.64,11.0,32627.0,6445.0,28566.0,6082.0,2.3087,<1H OCEAN
5518,-118.39,33.97,44.0,1097.0,186.0,513.0,185.0,6.2350,<1H OCEAN
11065,-117.88,33.79,32.0,1484.0,295.0,928.0,295.0,5.1418,<1H OCEAN
2317,-119.67,36.89,15.0,2373.0,364.0,1280.0,386.0,5.3080,INLAND
13868,-117.31,34.35,9.0,2404.0,390.0,1074.0,359.0,5.0198,INLAND
16457,-121.29,38.13,20.0,3168.0,514.0,1390.0,490.0,5.0000,INLAND


In [11]:
test_label = y.loc[test_data.index]
test_label

,median_house_value
14983,149600.0
8289,432500.0
10235,133900.0
15301,185400.0
9880,118800.0
5518,361400.0
11065,190300.0
2317,167500.0
13868,151900.0
16457,154800.0


In [12]:
test_data_prepared = full_pipeline.transform(test_data)
predicted = LR_model.predict(test_data_prepared)

In [13]:
predicted

array([ 182381.01348997,  272085.79947887,  206547.58012219,
        184211.32347503, -191207.51131105,  328538.91836362,
        262968.36885485,  172955.42599752,  165590.75444008,
        192452.91988579])

In [14]:
pd.DataFrame({'Bashorat': predicted, 'Asl qimat': test_label})

,Bashorat,Asl qimat
14983,182381.013490,149600.0
8289,272085.799479,432500.0
10235,206547.580122,133900.0
15301,184211.323475,185400.0
9880,-191207.511311,118800.0
5518,328538.918364,361400.0
11065,262968.368855,190300.0
2317,172955.425998,167500.0
13868,165590.754440,151900.0
16457,192452.919886,154800.0


Modelni Baholash

In [15]:
test_set

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
20046,-119.01,36.06,25.0,1505.0,NaN,1392.0,359.0,1.6812,47700.0,INLAND
3024,-119.46,35.14,30.0,2943.0,NaN,1565.0,584.0,2.5313,45800.0,INLAND
15663,-122.44,37.80,52.0,3830.0,NaN,1310.0,963.0,3.4801,500001.0,NEAR BAY
20484,-118.72,34.28,17.0,3051.0,NaN,1705.0,495.0,5.7376,218600.0,<1H OCEAN
9814,-121.93,36.62,34.0,2351.0,NaN,1063.0,428.0,3.7250,278000.0,NEAR OCEAN
...,...,...,...,...,...,...,...,...,...,...
15362,-117.22,33.36,16.0,3165.0,482.0,1351.0,452.0,4.6050,263300.0,<1H OCEAN
16623,-120.83,35.36,28.0,4323.0,886.0,1650.0,705.0,2.7266,266800.0,NEAR OCEAN
18086,-122.05,37.31,25.0,4111.0,538.0,1585.0,568.0,9.2298,500001.0,<1H OCEAN
2144,-119.76,36.77,36.0,2507.0,466.0,1227.0,474.0,2.7850,72300.0,INLAND


In [16]:
X_test = test_set.drop('median_house_value', axis=1)
y_test = test_set['median_house_value'].copy()
X_test_prepared = full_pipeline.transform(X_test)

In [17]:
y_predict = LR_model.predict(X_test_prepared)
y_predict

array([ 62006.78388667, 121643.79441299, 266990.00756469, ...,
       447883.67421698, 117237.3708917 , 185697.15157583])

In [18]:
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_test, y_predict)
print('MAE=', mae)

MAE= 50903.22718681366


In [19]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_predict)
print('MSE=', np.sqrt(mse))

MSE= 72717.21990877022


Random Forest

In [20]:
from sklearn.ensemble import RandomForestRegressor
RF_model = RandomForestRegressor()
RF_model.fit(X_prepared, y)

RandomForestRegressor()

In [21]:
y_predict = RF_model.predict(X_test_prepared)

In [22]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_predict)
print('MAE=', mae)
#

MAE= 50903.22718681366


Cross-Validation

In [23]:
X = df.drop('median_house_value', axis=1)
y = df['median_house_value'].copy()

x_prepared = full_pipeline.fit_transform(X)

In [24]:
from sklearn.model_selection import cross_val_score

ms_scores = cross_val_score(LR_model, x_prepared, y, scoring='neg_mean_squared_error', cv=5)
#

In [25]:
def display_scores(scores):
    print('Scores:', scores)
    print('Mean:', scores.mean())
    print('Std.dev:', scores.std())

In [26]:
display_scores(np.sqrt(-ms_scores))

Scores: [73397.21068666 74817.28199289 75436.69114751 76532.07576047
 66219.27245902]
Mean: 73280.50640930946
Std.dev: 3673.095693119273


In [27]:
scores = cross_val_score(RF_model, x_prepared, y, scoring='neg_mean_squared_error', cv=10)
LR_rmse_scores = np.sqrt(-scores)
display_scores(LR_rmse_scores)
#

Scores: [109982.49761657  49114.15363333  67185.03878965  59805.70537539
  62137.6186475   65816.14215406  50072.51230997  82146.86690111
  81280.55614277  53528.21408278]
Mean: 68106.93056531348
Std.dev: 17701.33540477844


Modelni saqlab olish

In [28]:
# pickle usuli
import pickle
filname = 'LR_model.pkl'
with open(filname, 'wb') as file:
    pickle.dump(LR_model, file)

In [29]:
with open(filname, 'rb') as file:
    LR_model = pickle.load(file)
# file ni oqish uchun ishlatiladi

Joblib usuli

In [31]:
import joblib
filename = 'RF_model.jbl'
joblib.dump(RF_model, filename)

['RF_model.jbl']

In [32]:
model = joblib.load(filename)
# filani chaqirib oldik

In [33]:
scores = cross_val_score(model, x_prepared, y, scoring='neg_mean_squared_error', cv=5)
LR_rmse_scores = np.sqrt(-scores)
display_scores(LR_rmse_scores)

Scores: [85866.4694169  67931.4491505  65877.95547615 95222.05896241
 67954.32808067]
Mean: 76570.45221732513
Std.dev: 11810.993401363297
